In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
import os
import pyodbc
from dotenv import load_dotenv
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose

In [4]:
customers = pd.read_csv(r"D:\All ML Projects\Retail_Demand_Forecasting\data\raw\customers.csv")
invoice_items = pd.read_csv(r"D:\All ML Projects\Retail_Demand_Forecasting\data\raw\invoice_items.csv")
products = pd.read_csv(r"D:\All ML Projects\Retail_Demand_Forecasting\data\raw\products.csv")
purchases = pd.read_csv(r"D:\All ML Projects\Retail_Demand_Forecasting\data\raw\purchases.csv")

In [5]:
purchases_clean = purchases.copy()

print("Original shape:", purchases.shape)
print("Cleaning copy shape:", purchases_clean.shape)

Original shape: (436689, 5)
Cleaning copy shape: (436689, 5)


In [6]:
print("Rows:", len(purchases_clean))
print("Columns:", purchases_clean.shape[1])

print("\nMissing values:")
print(purchases_clean.isnull().sum())

print("\nDuplicate rows:", purchases_clean.duplicated().sum())

Rows: 436689
Columns: 5

Missing values:
InvoiceID     0
date          0
CustomerID    0
product_id    0
quantity      0
dtype: int64

Duplicate rows: 5516


In [7]:
print("\nQuantity statistics:")
print(purchases_clean["quantity"].describe())


Quantity statistics:
count    436689.000000
mean         12.087628
std         172.252472
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max       80995.000000
Name: quantity, dtype: float64


In [8]:
purchases_clean.isnull().sum()

InvoiceID     0
date          0
CustomerID    0
product_id    0
quantity      0
dtype: int64

In [9]:
missing_values = purchases_clean.isnull().sum()

print(missing_values)

InvoiceID     0
date          0
CustomerID    0
product_id    0
quantity      0
dtype: int64


In [10]:
invalid_quantity = purchases_clean[
    purchases_clean["quantity"] <= 0
]

print("Invalid quantity rows:", len(invalid_quantity))

Invalid quantity rows: 0


In [11]:
print(purchases_clean["quantity"].min())

1


In [12]:
print("Missing InvoiceID:", purchases_clean["InvoiceID"].isnull().sum())
print("Missing CustomerID:", purchases_clean["CustomerID"].isnull().sum())
print("Missing product_id:", purchases_clean["product_id"].isnull().sum())

print("Invalid dates:", purchases_clean["date"].isna().sum())
print("Minimum date:", purchases_clean["date"].min())
print("Maximum date:", purchases_clean["date"].max())

Missing InvoiceID: 0
Missing CustomerID: 0
Missing product_id: 0
Invalid dates: 0
Minimum date: 2014-01-01
Maximum date: 2015-12-30


In [13]:
before = len(purchases_clean)

purchases_clean = purchases_clean[
    purchases_clean["quantity"] > 0
].copy()

after = len(purchases_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 436689
Rows after: 436689
Rows removed: 0


In [14]:
duplicates = purchases_clean[
    purchases_clean.duplicated(keep=False)
]

print("Duplicate rows:", len(duplicates))
print(duplicates.head(20))

Duplicate rows: 10627
     InvoiceID        date  CustomerID  product_id  quantity
476     536409  2014-12-01       17908        3040         1
480     536409  2014-12-01       17908        1608         1
485     536409  2014-12-01       17908        3669         1
508     536409  2014-12-01       17908        3669         1
512     536409  2014-12-01       17908        3062         1
518     536409  2014-12-01       17908        1608         1
528     536409  2014-12-01       17908        3062         1
530     536409  2014-12-01       17908        3040         1
539     536412  2014-12-01       17920        2989         1
546     536412  2014-12-01       17920        2989         1
547     536412  2014-12-01       17920        1238         1
556     536412  2014-12-01       17920           3         2
560     536412  2014-12-01       17920        1246         1
565     536412  2014-12-01       17920         754         1
569     536412  2014-12-01       17920           3         1
57

In [15]:
duplicate_groups = purchases_clean[
    purchases_clean.duplicated(keep=False)
].drop_duplicates()

print("Duplicate groups:", len(duplicate_groups))

Duplicate groups: 5111


In [16]:
purchases_clean = purchases_clean.drop_duplicates().copy()

In [17]:
duplicates.head(20)

,InvoiceID,date,CustomerID,product_id,quantity
476,536409,2014-12-01,17908,3040,1
480,536409,2014-12-01,17908,1608,1
485,536409,2014-12-01,17908,3669,1
508,536409,2014-12-01,17908,3669,1
512,536409,2014-12-01,17908,3062,1
518,536409,2014-12-01,17908,1608,1
528,536409,2014-12-01,17908,3062,1
530,536409,2014-12-01,17908,3040,1
539,536412,2014-12-01,17920,2989,1
546,536412,2014-12-01,17920,2989,1


In [18]:
duplicate_rows = purchases_clean[
    purchases_clean.duplicated(keep="first")
]

print("Duplicate rows to remove:", len(duplicate_rows))
print("Demand contained in duplicate rows:", duplicate_rows["quantity"].sum())

Duplicate rows to remove: 0
Demand contained in duplicate rows: 0


In [19]:
duplicate_demand = duplicate_rows["quantity"].sum()
total_demand = purchases_clean["quantity"].sum()

print(
    "Duplicate demand percentage:",
    round((duplicate_demand / total_demand) * 100, 2),
    "%"
)

Duplicate demand percentage: 0.0 %


In [20]:
before = len(purchases_clean)

purchases_clean = purchases_clean.drop_duplicates().copy()

after = len(purchases_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 431173
Rows after: 431173
Rows removed: 0


In [21]:
print("Remaining duplicate rows:",
      purchases_clean.duplicated().sum())

Remaining duplicate rows: 0


In [22]:
duplicate_rows_original = purchases[
    purchases.duplicated(keep="first")
]

duplicate_demand = duplicate_rows_original["quantity"].sum()
original_demand = purchases["quantity"].sum()

print("Duplicate rows removed:", len(duplicate_rows_original))
print("Demand removed:", duplicate_demand)
print(
    "Demand percentage removed:",
    round((duplicate_demand / original_demand) * 100, 2),
    "%"
)

Duplicate rows removed: 5516
Demand removed: 17225
Demand percentage removed: 0.33 %


In [23]:
invoice_product_counts = (
    purchases_clean
    .groupby(["InvoiceID", "product_id"])
    .size()
    .reset_index(name="row_count")
)

print("Invoice + product groups:", len(invoice_product_counts))

print(
    "Groups appearing more than once:",
    (invoice_product_counts["row_count"] > 1).sum()
)

Invoice + product groups: 425701
Groups appearing more than once: 5292


In [24]:
repeated_invoice_product = invoice_product_counts[
    invoice_product_counts["row_count"] > 1
]

print(repeated_invoice_product.head(20))

      InvoiceID  product_id  row_count
39           18        2956          2
61           27        2956          2
75           32        1341          2
88           34        4004          2
296         115        2304          2
346         133        4004          2
426         163        4004          2
485         187        3280          2
492         190        4004          2
522         203        3426          2
665         262        2662          2
800         314        3891          2
1235        475         883          2
1301        502        2304          2
1352        523        3891          2
1389        539        4004          2
1496        578        2956          2
1610        626         522          2
1774        686        3426          2
1794        695        2961          2


In [25]:
purchases_clean.merge(
    repeated_invoice_product,
    on=["InvoiceID", "product_id"]
).head(30)

,InvoiceID,date,CustomerID,product_id,quantity,row_count
0,536381,2014-12-01,15311,2486,1,2
1,536381,2014-12-01,15311,2486,3,2
2,536409,2014-12-01,17908,84,3,3
3,536409,2014-12-01,17908,342,1,2
4,536409,2014-12-01,17908,84,1,3
5,536409,2014-12-01,17908,342,5,2
6,536409,2014-12-01,17908,84,2,3
7,536412,2014-12-01,17920,1238,1,2
8,536412,2014-12-01,17920,1232,3,2
9,536412,2014-12-01,17920,3130,1,2


In [26]:
extreme_rows = purchases_clean[
    purchases_clean["quantity"] > 1000
].sort_values(
    "quantity",
    ascending=False
)

print(extreme_rows.head(20))

        InvoiceID        date  CustomerID  product_id  quantity
397451     581483  2015-12-09       16446        2414     80995
37126      541431  2015-01-18       12346        2074     74215
370473     578841  2015-11-25       13256         239     12540
308128     573008  2015-10-27       12901        3941      4800
143243     554868  2015-05-27       13135        3393      4300
61586      544612  2015-02-22       18087        1159      3906
188812     560599  2015-07-19       14609        1193      3186
109624     550461  2015-04-18       15749        1198      3114
32732      540815  2015-01-11       15749        1198      3114
318104     573995  2015-11-02       16308        3365      3000
204041     562439  2015-08-04       12931         209      2880
3670       536830  2014-12-02       16754        3941      2880
139695     554272  2015-05-23       12901        2361      2700
49734      543057  2015-02-03       16333        3941      2592
57506      544152  2015-02-16       1460

In [27]:
max_row = purchases_clean.loc[
    purchases_clean["quantity"].idxmax()
]

print(max_row)

InvoiceID         581483
date          2015-12-09
CustomerID         16446
product_id          2414
quantity           80995
Name: 397451, dtype: object


In [28]:
for threshold in [100, 500, 1000, 5000, 10000]:
    count = (purchases_clean["quantity"] > threshold).sum()
    demand = purchases_clean.loc[
        purchases_clean["quantity"] > threshold,
        "quantity"
    ].sum()

    print(
        f"> {threshold}: "
        f"{count:,} rows | "
        f"{demand:,} units"
    )

> 100: 4,662 rows | 1,393,806 units
> 500: 407 rows | 542,586 units
> 1000: 105 rows | 337,153 units
> 5000: 3 rows | 167,750 units
> 10000: 3 rows | 167,750 units


In [29]:
extreme_transactions = purchases_clean[
    purchases_clean["quantity"] > 10000
].copy()

print(extreme_transactions)

        InvoiceID        date  CustomerID  product_id  quantity
37126      541431  2015-01-18       12346        2074     74215
370473     578841  2015-11-25       13256         239     12540
397451     581483  2015-12-09       16446        2414     80995


In [30]:
extreme_customers = purchases_clean[
    purchases_clean["CustomerID"].isin(
        extreme_transactions["CustomerID"]
    )
]

print(
    extreme_customers
    .groupby("CustomerID")["quantity"]
    .agg(["count", "sum", "mean", "max"])
)

            count    sum     mean    max
CustomerID                              
12346           1  74215  74215.0  74215
13256           1  12540  12540.0  12540
16446           3  80997  26999.0  80995


In [31]:
extreme_products = purchases_clean[
    purchases_clean["product_id"].isin(
        extreme_transactions["product_id"]
    )
]

print(
    extreme_products
    .groupby("product_id")["quantity"]
    .agg(["count", "sum", "mean", "max"])
)

            count    sum          mean    max
product_id                                   
239            33  13699    415.121212  12540
2074          198  77916    393.515152  74215
2414            1  80995  80995.000000  80995


In [32]:
daily_demand = (
    purchases_clean
    .groupby("date")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

print(daily_demand.head(10))

date
2015-12-09    90655
2015-01-18    80762
2015-10-05    45982
2015-09-20    42711
2015-12-07    41091
2015-10-20    40938
2015-08-04    40258
2015-12-05    38463
2015-11-23    38422
2015-11-10    38135
Name: quantity, dtype: int64


In [33]:
extreme_anomalies = purchases_clean[
    purchases_clean["quantity"].isin([74215, 80995])
]

print(extreme_anomalies)

        InvoiceID        date  CustomerID  product_id  quantity
37126      541431  2015-01-18       12346        2074     74215
397451     581483  2015-12-09       16446        2414     80995


In [34]:
before = len(purchases_clean)
demand_before = purchases_clean["quantity"].sum()

purchases_clean = purchases_clean[
    ~purchases_clean["quantity"].isin([74215, 80995])
].copy()

after = len(purchases_clean)
demand_after = purchases_clean["quantity"].sum()

print("Rows removed:", before - after)
print("Demand removed:", demand_before - demand_after)

Rows removed: 2
Demand removed: 155210


In [35]:
print("Final shape:", purchases_clean.shape)

print("\nMissing values:")
print(purchases_clean.isnull().sum())

print("\nExact duplicates:",
      purchases_clean.duplicated().sum())

print("\nInvalid quantities:",
      (purchases_clean["quantity"] <= 0).sum())

print("\nQuantity statistics:")
print(purchases_clean["quantity"].describe())

Final shape: (431171, 5)

Missing values:
InvoiceID     0
date          0
CustomerID    0
product_id    0
quantity      0
dtype: int64

Exact duplicates: 0

Invalid quantities: 0

Quantity statistics:
count    431171.000000
mean         11.842399
std          45.426099
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max       12540.000000
Name: quantity, dtype: float64


In [36]:
print("\nLargest quantities:")
print(
    purchases_clean
    .nlargest(10, "quantity")
    [["InvoiceID", "date", "CustomerID", "product_id", "quantity"]]
)


Largest quantities:
        InvoiceID        date  CustomerID  product_id  quantity
370473     578841  2015-11-25       13256         239     12540
308128     573008  2015-10-27       12901        3941      4800
143243     554868  2015-05-27       13135        3393      4300
61586      544612  2015-02-22       18087        1159      3906
188812     560599  2015-07-19       14609        1193      3186
32732      540815  2015-01-11       15749        1198      3114
109624     550461  2015-04-18       15749        1198      3114
318104     573995  2015-11-02       16308        3365      3000
3670       536830  2014-12-02       16754        3941      2880
204041     562439  2015-08-04       12931         209      2880


In [37]:
original_rows = len(purchases)
clean_rows = len(purchases_clean)

original_demand = purchases["quantity"].sum()
clean_demand = purchases_clean["quantity"].sum()

print("BEFORE CLEANING")
print("----------------")
print("Rows:", original_rows)
print("Total demand:", original_demand)

print("\nAFTER CLEANING")
print("----------------")
print("Rows:", clean_rows)
print("Total demand:", clean_demand)

print("\nCHANGES")
print("----------------")
print("Rows removed:", original_rows - clean_rows)
print("Demand removed:", original_demand - clean_demand)

print(
    "Rows removed (%):",
    round((original_rows - clean_rows) / original_rows * 100, 2)
)

print(
    "Demand removed (%):",
    round((original_demand - clean_demand) / original_demand * 100, 2)
    
)

BEFORE CLEANING
----------------
Rows: 436689
Total demand: 5278534

AFTER CLEANING
----------------
Rows: 431171
Total demand: 5106099

CHANGES
----------------
Rows removed: 5518
Demand removed: 172435
Rows removed (%): 1.26
Demand removed (%): 3.27


In [38]:
print("\nQuantity comparison")

print("\nBEFORE:")
print(purchases["quantity"].describe())

print("\nAFTER:")
print(purchases_clean["quantity"].describe())


Quantity comparison

BEFORE:
count    436689.000000
mean         12.087628
std         172.252472
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max       80995.000000
Name: quantity, dtype: float64

AFTER:
count    431171.000000
mean         11.842399
std          45.426099
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max       12540.000000
Name: quantity, dtype: float64


### SAVE THE CLEANED DATASET

In [42]:
import os

os.makedirs("../data/processed", exist_ok=True)

clean_path = "../data/processed/purchases_clean.csv"

purchases_clean.to_csv(clean_path, index=False)

print(f"Saved to: {os.path.abspath(clean_path)}")

Saved to: d:\All ML Projects\Retail_Demand_Forecasting\data\processed\purchases_clean.csv


In [43]:
import os

print("File exists:", os.path.exists(clean_path))
print("Path:", os.path.abspath(clean_path))

File exists: True
Path: d:\All ML Projects\Retail_Demand_Forecasting\data\processed\purchases_clean.csv


In [44]:
clean_check = pd.read_csv(clean_path)

print("Loaded shape:", clean_check.shape)
print("Columns:", clean_check.columns.tolist())

Loaded shape: (431171, 5)
Columns: ['InvoiceID', 'date', 'CustomerID', 'product_id', 'quantity']
